In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from rich.console import Console
from rich.table import Table
from rich.live import Live
from rich.panel import Panel
from rich.columns import Columns
from rich.markdown import Markdown

In [3]:
import torch
import torchtext

In [4]:
console = Console()

In [5]:
KNOWLEDGE_BASE = """
1.1 WAKE UP AT A FIXED TIME
Set a consistent wake-up time and keep it every day, including weekends. A stable schedule trains your body clock so you wake feeling rested instead of groggy. Place the alarm across the room so you must physically get out of bed to switch it off. Avoid hitting snooze, as the short fragmented sleep it gives is low quality and leaves you more tired. The moment your feet touch the floor, the day has started.

1.2 GET SUNLIGHT EARLY
Within the first 30 minutes of waking, expose yourself to natural daylight. Open the curtains, step onto a balcony, or take a short walk outside. Morning light signals your brain to stop producing the sleep hormone melatonin and helps set your circadian rhythm for the day. This single habit improves alertness in the morning and makes it easier to fall asleep at night.

1.3 HYDRATE BEFORE CAFFEINE
You lose water through breathing and sweating during the night, so you wake up mildly dehydrated. Drink a full glass of water before reaching for coffee or tea. Hydration restores focus, reduces headaches, and kick-starts your metabolism. If you want, add a pinch of salt or a squeeze of lemon to help your body absorb the water. Delay caffeine by 60 to 90 minutes to avoid an early energy crash.

1.4 MOVE YOUR BODY
Spend 5 to 15 minutes on light physical activity such as stretching, yoga, a brisk walk, or a few bodyweight exercises. Movement increases blood flow, raises your core temperature, and releases endorphins that lift your mood. You do not need an intense workout; the goal is to wake the body gently and shake off stiffness from sleeping. Consistency matters far more than intensity.

1.5 EAT A BALANCED BREAKFAST
Fuel your body with a breakfast that combines protein, healthy fats, and fibre to keep energy steady until lunch. Good options include eggs, oats, yoghurt with fruit, or whole-grain toast with nut butter. Avoid sugary cereals and pastries, which spike blood sugar and lead to a mid-morning slump. Eating well in the morning improves concentration and reduces unhealthy snacking later in the day.

1.6 PLAN YOUR DAY
Take five quiet minutes to review your schedule and choose the three most important tasks for the day. Writing them down clears mental clutter and gives you a clear direction before distractions arrive. Tackle the hardest or most important task first, while your willpower and focus are at their peak. A small amount of planning prevents the day from running away from you.

1.7 AVOID YOUR PHONE FIRST
Resist the urge to check messages, email, or social media the moment you wake. Starting the day reacting to other people's demands puts you in a stressed, scattered state. Give yourself at least the first 30 minutes phone-free so your mind can settle and you can act on your own priorities. The notifications will still be there once your morning routine is done.
"""

In [6]:
chunks = KNOWLEDGE_BASE.strip().split("\n\n")
chunks

['1.1 WAKE UP AT A FIXED TIME\nSet a consistent wake-up time and keep it every day, including weekends. A stable schedule trains your body clock so you wake feeling rested instead of groggy. Place the alarm across the room so you must physically get out of bed to switch it off. Avoid hitting snooze, as the short fragmented sleep it gives is low quality and leaves you more tired. The moment your feet touch the floor, the day has started.',
 '1.2 GET SUNLIGHT EARLY\nWithin the first 30 minutes of waking, expose yourself to natural daylight. Open the curtains, step onto a balcony, or take a short walk outside. Morning light signals your brain to stop producing the sleep hormone melatonin and helps set your circadian rhythm for the day. This single habit improves alertness in the morning and makes it easier to fall asleep at night.',
 '1.3 HYDRATE BEFORE CAFFEINE\nYou lose water through breathing and sweating during the night, so you wake up mildly dehydrated. Drink a full glass of water b

In [7]:
vectorize = TfidfVectorizer()
chunks_vectors = vectorize.fit_transform(chunks)
chunks_vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 390 stored elements and shape (7, 292)>

In [8]:
def retrieve(query: str, k: int = 1):
    query_vector = vectorize.transform([query])

    similarity = cosine_similarity(query_vector, chunks_vectors).flatten()

    top_k_indices = np.argsort(similarity)[::-1][:k]

    return [chunks[i] for i in top_k_indices],similarity


In [9]:
def main():
    example_query = [
        "When should I wake up?",
        "Should I check my phone in the morning?",
        "What should I eat for breakfast?"
    ]

    for i, query in enumerate(example_query, 1):
        retrieve_chunks, scores = retrieve(query, k = 2)

        table = Table(title="Retrieval Results")
        table.add_column("Rank")
        table.add_column("Similarity Score")
        table.add_column("Retrieval Text", width=60)

        for j, (chunk, score) in enumerate(zip(retrieve_chunks , scores)):
            table.add_row(
                str(j + 1),
                f"{score: .3f}",
                chunk[:200] + "..." if len(chunk) > 200 else chunk,
            )

        console.print(table)
        
main()

                                    Retrieval Results                                     
┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Similarity Score ┃ Retrieval Text                                               ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │  0.283           │ 1.1 WAKE UP AT A FIXED TIME                                  │
│      │                  │ Set a consistent wake-up time and keep it every day,         │
│      │                  │ including weekends. A stable schedule trains your body clock │
│      │                  │ so you wake feeling rested instead of groggy. Place the      │
│      │                  │ al...                                                        │
│ 2    │  0.000           │ 1.3 HYDRATE BEFORE CAFFEINE                                  │
│      │                  │ You lose water through breathing and sweating during the     │
│      │                  │ night, so you wake up mildly dehydrated. Drink a full glass  │
│      │                  │ of water before reaching for coffee or tea. Hydration r...   │
└──────┴──────────────────┴──────────────────────────────────────────────────────────────┘

                                    Retrieval Results                                     
┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Similarity Score ┃ Retrieval Text                                               ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │  0.069           │ 1.7 AVOID YOUR PHONE FIRST                                   │
│      │                  │ Resist the urge to check messages, email, or social media    │
│      │                  │ the moment you wake. Starting the day reacting to other      │
│      │                  │ people's demands puts you in a stressed, scattered state.    │
│      │                  │ G...                                                         │
│ 2    │  0.189           │ 1.2 GET SUNLIGHT EARLY                                       │
│      │                  │ Within the first 30 minutes of waking, expose yourself to    │
│      │                  │ natural daylight. Open the curtains, step onto a balcony, or │
│      │                  │ take a short walk outside. Morning light signals your        │
│      │                  │ brai...                                                      │
└──────┴──────────────────┴──────────────────────────────────────────────────────────────┘

                                    Retrieval Results                                     
┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Similarity Score ┃ Retrieval Text                                               ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │  0.000           │ 1.5 EAT A BALANCED BREAKFAST                                 │
│      │                  │ Fuel your body with a breakfast that combines protein,       │
│      │                  │ healthy fats, and fibre to keep energy steady until lunch.   │
│      │                  │ Good options include eggs, oats, yoghurt with fruit, or w... │
│ 2    │  0.045           │ 1.2 GET SUNLIGHT EARLY                                       │
│      │                  │ Within the first 30 minutes of waking, expose yourself to    │
│      │                  │ natural daylight. Open the curtains, step onto a balcony, or │
│      │                  │ take a short walk outside. Morning light signals your        │
│      │                  │ brai...                                                      │
└──────┴──────────────────┴──────────────────────────────────────────────────────────────┘

In [10]:
def retrieve(query: str, k: int = 1):
    query_vector = vectorize.transform([query])

    similarity = cosine_similarity(query_vector, chunks_vectors).flatten()

    top_k_indices = np.argsort(similarity)[::-1][:k]

    return [chunks[i] for i in top_k_indices]


In [11]:
import ollama

In [12]:
model = "qwen3-vl:4b"
TEMPRATURE = 0.0

In [13]:
RAG_PROMPT = """Use the following context to answer the question. If you cannot find the answer in the context say "I can't find the answer to this in the given context"  

<contexts>
{context}
</contexts>

<question>
{query}
</question>

""".strip()

resp = ollama.chat(model=model, messages=[
    {"role": "user", "content": "Explain RAG in one line"}
])

answer_text = resp["message"]["content"]
console.print(Panel(Markdown(answer_text), title="Answer", border_style="green"))

In [14]:
from langchain.chat_models import init_chat_model

In [15]:
llm = init_chat_model(model=model, model_provider="ollama", temprature = TEMPRATURE,max_tokens = 250, timout = 30)

In [16]:
def answer(query: str, k: int = 2):
    context_chunks = retrieve(query, 2)
    context_string = [f"<context> {c} >/context>" for c in context_chunks]
    for chunk in llm.stream(RAG_PROMPT.format(context = context_string, query = query)):
        yield chunk.content

        

In [17]:
def ask_question(query: str):

    full_response = ""
    thinking_content = ""
    answer_content = ""
    in_thinking = False

    with Live(console=console, refresh_per_second=8) as live:
        for content in answer(query):
            full_response += content

            if "<think>" in content:
                in_thinking = True
                content = content.replace("<think>", "")

            if in_thinking:
                if "</think>" in content:
                    thinking_content += content.replace("</think>", "")
                    in_thinking = False
                else:
                    thinking_content += content
            else:
                answer_content += content

            panels = []

            if thinking_content.strip():
                thinking_panel = Panel(
                    Markdown(thinking_content),
                    title="Thinking",
                    border_style="blue",
                )
                panels.append(thinking_panel)

            if answer_content.strip():
                answer_panel = Panel(
                    Markdown(answer_content),
                    title="Answer",
                    border_style="green",
                )
                panels.append(answer_panel)

            live.update(Columns(panels, expand=True,width=55))


In [18]:
query = "Explain the most importatnt Thing i should do?"
console.print(Panel(query, title="Query", border_style="blue", width=60))
retrieved_doc = retrieve(query)
console.print(Panel(retrieved_doc[0],
                    title="Retrieved text"),width=60)

╭───────────────────────── Query ──────────────────────────╮
│ Explain the most importatnt Thing i should do?           │
╰──────────────────────────────────────────────────────────╯

╭───────────────────── Retrieved text ─────────────────────╮
│ 1.6 PLAN YOUR DAY                                        │
│ Take five quiet minutes to review your schedule and      │
│ choose the three most important tasks for the day.       │
│ Writing them down clears mental clutter and gives you a  │
│ clear direction before distractions arrive. Tackle the   │
│ hardest or most important task first, while your         │
│ willpower and focus are at their peak. A small amount of │
│ planning prevents the day from running away from you.    │
╰──────────────────────────────────────────────────────────╯

In [19]:
ask_question(query)

Output()